In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

c:\uni\seriousism\reminisGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
artists_texts = pd.read_csv("../data/processed/artist_texts.csv")
artists_texts

,artist_id,artist_name,text_for_embedding
0,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,"Albums: Ultramega OK. Tags: grunge, alternativ..."
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,"Albums: Bleach. Tags: grunge, alternative rock..."
2,7527f6c2-d762-4b88-b5e2-9244f1e34c46,Deftones,"Albums: Adrenaline. Tags: alternative metal, n..."
3,83b9cbe7-9857-49e2-ab8e-b57b01038103,Pearl Jam,"Albums: Ten. Tags: grunge, alternative rock, r..."
4,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,"Albums: Gish. Tags: alternative rock, dream po..."
5,ba853904-ae25-4ebb-89d6-c44cfbd71bd2,Blur,"Albums: Leisure. Tags: alternative rock, britp..."


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = artists_texts['text_for_embedding'].tolist()
embeddings = model.encode(texts, convert_to_numpy=True)

embeddings.shape

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3487.61it/s]


(6, 384)

In [7]:
out_dir = Path("../data/processed/embeddings")
out_dir.mkdir(parents=True, exist_ok=True)

np.save(out_dir / "artist_embeddings.npy", embeddings)

artists_texts.to_csv(out_dir / "artist_embedding_metadata.csv", index=False)

#### Sanity Check

In [8]:
loaded_embeddings = np.load(out_dir / "artist_embeddings.npy")
metadata = pd.read_csv(out_dir / "artist_embedding_metadata.csv")

loaded_embeddings.shape, metadata.shape

((6, 384), (6, 3))

#### Similarity Matrix Check

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

similarity_df = pd.DataFrame(
  similarity_matrix,
  index=artists_texts['artist_name'],
  columns=artists_texts['artist_name']
)

similarity_df

artist_name,Soundgarden,Nirvana,Deftones,Pearl Jam,The Smashing Pumpkins,Blur
artist_name,,,,,,
Soundgarden,1.000000,0.711962,0.681799,0.740760,0.609996,0.684405
Nirvana,0.711962,1.000000,0.568362,0.754655,0.629098,0.675207
Deftones,0.681799,0.568362,1.000000,0.653498,0.595093,0.691868
Pearl Jam,0.740760,0.754655,0.653498,1.000000,0.663584,0.760920
The Smashing Pumpkins,0.609996,0.629098,0.595093,0.663584,1.000000,0.696780
Blur,0.684405,0.675207,0.691868,0.760920,0.696780,1.000000


In [19]:
def get_top_similar(artist_name, top_k=3):
    scores = similarity_df[artist_name].sort_values(ascending=False)
    scores = scores.drop(artist_name)
    return scores.head(top_k)

In [20]:
topk_rows = []

for artist in similarity_df.index:
    top_scores = get_top_similar(artist, top_k=3)
    
    for rank, (candidate, score) in enumerate(top_scores.items(), start=1):
        topk_rows.append({
            "query_artist": artist,
            "rank": rank,
            "retrieved_artist": candidate,
            "similarity_score": score
        })

topk_df = pd.DataFrame(topk_rows)
topk_df

,query_artist,rank,retrieved_artist,similarity_score
0,Soundgarden,1,Pearl Jam,0.740760
1,Soundgarden,2,Nirvana,0.711962
2,Soundgarden,3,Blur,0.684405
3,Nirvana,1,Pearl Jam,0.754655
4,Nirvana,2,Soundgarden,0.711962
5,Nirvana,3,Blur,0.675207
6,Deftones,1,Blur,0.691868
7,Deftones,2,Soundgarden,0.681799
8,Deftones,3,Pearl Jam,0.653498
9,Pearl Jam,1,Blur,0.760920


In [21]:
pairs = []

for i, artist_a in enumerate(similarity_df.index):
    for j, artist_b in enumerate(similarity_df.columns):
        if i < j:
            pairs.append({
                "artist_a": artist_a,
                "artist_b": artist_b,
                "similarity": similarity_df.loc[artist_a, artist_b]
            })

pair_df = pd.DataFrame(pairs).sort_values("similarity", ascending=False)
pair_df

,artist_a,artist_b,similarity
13,Pearl Jam,Blur,0.760920
6,Nirvana,Pearl Jam,0.754655
2,Soundgarden,Pearl Jam,0.740760
0,Soundgarden,Nirvana,0.711962
14,The Smashing Pumpkins,Blur,0.696780
11,Deftones,Blur,0.691868
4,Soundgarden,Blur,0.684405
1,Soundgarden,Deftones,0.681799
8,Nirvana,Blur,0.675207
12,Pearl Jam,The Smashing Pumpkins,0.663584


In [22]:
topk_df.to_csv("../data/processed/artist_vector_topk.csv", index=False)

## Initial Vector Retrieval Result

The first vector similarity check produced reasonable artist-level retrieval results based on album titles and MusicBrainz tags.

The highest similarity pair is **Pearl Jam** and **Blur** with a cosine similarity score of **0.760920**. This suggests that the vector baseline is mostly driven by overlapping or semantically related tags, rather than historical membership relationships.

This is useful for the project because it shows that vector retrieval and graph retrieval may surface different types of connections:

- Vector retrieval: genre, style, and tag-based similarity
- Graph retrieval: historical and relational connections, such as shared band members

Because member names were intentionally excluded from the embedding text, these vector results do not leak graph-only relationship information.

A concrete example of this difference appears in the initial results: Pearl Jam is more similar to Blur in the vector baseline (**0.760920**) than to Soundgarden (**0.740760**), even though Pearl Jam and Soundgarden have stronger graph-relevant evidence through shared membership.

This supports the project’s core comparison: vector retrieval captures semantic/tag overlap, while graph retrieval is expected to better capture relational and historical connections.